# PFAS Groundwater — V2: Non-Destructive Fusion Variants (T1a)

**Task T1a**: predict EPA-2024 regulatory PFAS exceedance (binary) from context features only  
(strict predictive mode, no PFAS measurement as a feature, no lat/lon, C-LOC.1).

**This notebook is AUTONOMOUS** (CLAUDE.md §4): it bootstraps `src/` and the versioned  
dataset via `git clone` (no Google Drive, no gdown), installs PyTorch Geometric for the  
Colab torch wheel, then runs the full V2 experiment via `V2.run(smoke=...)`.

**V2 research question:**  
V0 embedding fusion compressed the 64-D HGT embedding with PCA-to-95%-variance (~47-48  
components kept), yet the OOF AUC (0.667) fell below the XGB-tabular in-run wall (0.688).  
Was PCA the bottleneck, or is the HGT embedding simply not additive for XGBoost on this task?  
V2 tests **five non-destructive fusions** over the same OOF backbone:
- `v2a_full64d` — XGBoost on raw [tabular || embedding-64D] (no PCA at all)
- `v2b_pca8` — XGBoost on [tabular || PCA(embedding, k=8)]
- `v2b_pca16` — XGBoost on [tabular || PCA(embedding, k=16)]
- `v2c_gating` — learned gating MLP (softly weights graph vs tabular, OOF nested-LOBO)
- `v2c_gating_xgb` — XGBoost on the gating head fused representation

Plus three **reference columns** computed in the same run:  
`xgb_seul` (backbone XGB), `stacking_v0` (meta-XGB), `fusion_pca95_v0` (V0 baseline).

**Anti-leak design:** nested LOBO on k=8 KMeans spatial blocks; cross-block edges = 0 (asserted);  
inductive HGT; no lat/lon; thresholds from OOF/VAL only (C-THR). Random arm (Δ) computed to  
measure spatial-leakage inflation.

**Training-curve output (CLAUDE.md §3.8):** Variant (c) gating MLP is iterative — per-epoch  
train_loss and val_auc curves are written to `figures/gating_training_curves.png`.  
Variants (a), (b), (c-xgb) are single-fit XGBoost — per-fold AUC bar chart in  
`figures/fold_auc_comparison.png` is the fold-level diagnostic.  
The scatter `figures/spatial_vs_random_scatter.png` shows the spatial-inflation profile.

**Runtime:** full run (backbone HGT built 2x: spatial + random) estimated **25-35 min on Colab T4 GPU**.

---
**CRITICAL — push the code BEFORE running on Colab (see Cell 0 below).**  
`src/v2_fusion_t1.py` (new), `src/v2_fusion_gating.py` (new), and  
`src/hgt_fusion_stacking_t1.py` (modified) are **not yet committed**.  
Colab only sees committed + pushed files.  
Push first, then set `GIT_REF` to the resulting branch/commit.

## Cell 0 — BEFORE YOU RUN ON COLAB: push the code to the remote

The following files are **not yet committed** (or locally modified) in git:
- `src/v2_fusion_t1.py` — new V2 runner (5 fusion variants + references + figures)
- `src/v2_fusion_gating.py` — new gating MLP head (variant c, §3.8 curves)
- `src/hgt_fusion_stacking_t1.py` — modified (accessor + OOFArrays updates)
- `notebooks/v2_fusion_colab.ipynb` — this notebook

**Colab only sees what is committed and pushed to `REPO_URL`.** Run the following  
locally **before** opening this notebook on Colab:

```bash
git add src/v2_fusion_t1.py src/v2_fusion_gating.py src/hgt_fusion_stacking_t1.py notebooks/v2_fusion_colab.ipynb
git commit -m 'feat(v2): non-destructive fusion variants for T1a, gating MLP §3.8'
git push
```

Then set `GIT_REF` in the parameter cell below to the branch (e.g. `"main"`) or the  
resulting commit SHA. **Do not hardcode a stale SHA** — use `"main"` unless pinning  
for reproducibility.

The anti-stale-code guard in Cell 3 will stop the notebook with an explicit error if the  
cloned code does not contain the required symbols.

In [ ]:
# ============================================================
# USER PARAMETERS — adjust before running
# ============================================================

SMOKE_TEST = False        # True = fast CPU sanity check (~3 min); False = full GPU run

# IMPORTANT: set GIT_REF to the branch/commit that contains:
#   src/v2_fusion_t1.py (new), src/v2_fusion_gating.py (new),
#   src/hgt_fusion_stacking_t1.py (modified)
# See the markdown cell above for the exact git commands.
REPO_URL = "https://github.com/dnwiloic/pfas-gnn.git"
GIT_REF  = "main"        # branch name or full commit SHA
DATA_PATH = "data/CA-PFAS-ASGWS.parquet"  # relative to repo root (versioned in repo)

EXP_DIR      = "experiments/v2_fusion"          # full-run artefacts
SMOKE_EXP_DIR = "experiments/v2_fusion_smoke"   # smoke artefacts (separate)

# Full-run hyperparameters (match src/v2_fusion_t1.py defaults)
FULL_N_BLOCKS    = 8      # spatial KMeans CV blocks
FULL_HIDDEN      = 64     # HGT backbone hidden dim
FULL_LAYERS      = 2
FULL_DROPOUT     = 0.3
FULL_HEADS       = 4      # HGT attention heads
FULL_MAX_EPOCHS  = 400
FULL_PATIENCE    = 50
FULL_LR          = 5e-3
FULL_WEIGHT_DECAY = 5e-4
COMPUTE_DELTA    = True   # build random OOF arm (Δ) to measure spatial-leakage inflation

SEED = 42

# ============================================================
# DURATION ESTIMATE (SMOKE_TEST=False, Colab T4 GPU)
# ============================================================
# V2 builds the HGT backbone TWICE (spatial + random arm for Δ).
# Each backbone = 8-fold HGT + 8-fold XGB + 8-fold LGBM ≈ 12-18 min on Colab T4.
# Fusion variant heads (a, b, c) are lightweight on top of the OOF arrays.
# Gating MLP training adds ~3-5 min (CPU, no GPU needed for the MLP).
# Total estimate with COMPUTE_DELTA=True: ~25-35 min on T4 GPU.
# With COMPUTE_DELTA=False: ~12-18 min (single backbone).
#
# Checkpointing: metrics.json written at end of run(); gating history and figures
# written incrementally by V2g.plot_gating_curves.
# ============================================================

print("Parameters set.")
print(f"  SMOKE_TEST   : {SMOKE_TEST}")
print(f"  REPO_URL     : {REPO_URL}")
print(f"  GIT_REF      : {GIT_REF}")
print(f"  DATA_PATH    : {DATA_PATH}")
print(f"  EXP_DIR      : {EXP_DIR}")
print(f"  COMPUTE_DELTA: {COMPUTE_DELTA}")
if not SMOKE_TEST:
    print()
    print("  Full-run estimate (Colab T4 GPU): ~25-35 min (COMPUTE_DELTA=True)")
    print("  Backbone HGT built 2x (spatial + random): each ~12-18 min")
    print("  Fusion heads (a, b, c) + gating MLP: ~3-5 min on top of OOF arrays")

## Cell 1 — GPU detection & runtime versions

**Note on runtime choice:**  
- The HGT backbone GNN training is GPU-accelerated (T4 recommended for the full run).  
- The XGBoost fusion heads and gating MLP run on CPU regardless of runtime.  
- For `SMOKE_TEST=True`, a CPU runtime (High-RAM) is fine and avoids GPU quota warnings.  
- For the full run, select **T4 GPU** via: Runtime > Change runtime type > T4 GPU.

In [ ]:
import sys, platform, subprocess

print(f"Python  : {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")

IN_COLAB = False
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    pass
print(f"IN_COLAB: {IN_COLAB}")

try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"torch   : {torch.__version__}  CUDA avail: {cuda_ok}")
    if cuda_ok:
        print(f"GPU     : {torch.cuda.get_device_name(0)}")
        print(f"CUDA    : {torch.version.cuda}")
    else:
        print("WARNING: no GPU detected.")
        if SMOKE_TEST:
            print("  SMOKE_TEST=True -> CPU run is expected and fine.")
        else:
            print("  SMOKE_TEST=False -> HGT backbone needs GPU (~25-35 min on T4).")
            print("  Go to: Runtime > Change runtime type > T4 GPU")
except ImportError:
    print("torch not yet installed (will be available after Cell 2 installs).")

print()
print("Note: XGBoost fusion heads and gating MLP use CPU regardless of runtime.")
print("Note: Only the HGT backbone GNN training benefits from GPU.")

## Cell 2 — Install pinned dependencies + verify imports

PyG compiled extensions must match `torch.__version__` + CUDA tag.  
We detect both at runtime and install from the matching wheel index.  
This cell is idempotent: if PyG is already importable it skips the heavy install.

**V2-specific imports verified:** `dropout_edge`, `GraphNorm`, `SAGEConv`, `HeteroConv`, `HGTConv`  
(required by `src/hgt_fusion_stacking_t1.py` and `src/v2_fusion_gating.py`).

In [ ]:
import subprocess, sys

# --- PyTorch Geometric (must match runtime torch + CUDA) ---
def ensure_pyg():
    try:
        import torch_geometric
        print(f"torch_geometric already present: {torch_geometric.__version__}")
        return
    except ImportError:
        pass
    import torch
    tv = torch.__version__.split("+")[0]
    cuda = torch.version.cuda
    tag = f"cu{cuda.replace('.', '')}" if (cuda and torch.cuda.is_available()) else "cpu"
    idx = f"https://data.pyg.org/whl/torch-{tv}+{tag}.html"
    print(f"Installing torch_geometric for torch={tv}, device={tag}")
    print(f"  Wheel index: {idx}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch_geometric", "-f", idx], check=True)
    # Optional compiled helpers (scatter/sparse) — ignore failure on CPU
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch_scatter", "torch_sparse", "-f", idx])

ensure_pyg()

# --- V2-specific import verification ---
# These symbols are required by the HGT backbone (hgt_fusion_stacking_t1 -> gnn_hetero_t1)
# and by the V2 gating head (v2_fusion_gating.py). Verify ALL before proceeding.
import torch_geometric
print(f"PyG: {torch_geometric.__version__}")

from torch_geometric.utils import dropout_edge      # used in gnn_hetero_t1 encoder
print("  dropout_edge import OK")
from torch_geometric.nn import GraphNorm            # used in gnn_hetero_t1 encoder
print("  GraphNorm import OK")
from torch_geometric.nn import SAGEConv             # used in gnn_hetero_t1 encoder
print("  SAGEConv import OK")
from torch_geometric.nn import HeteroConv           # used in gnn_hetero_t1 encoder
print("  HeteroConv import OK")
from torch_geometric.nn import HGTConv              # HGT backbone (reference encoder)
print("  HGTConv import OK")
print()
print("All V2 PyG symbols verified.")

# --- XGBoost (fusion aval heads + in-run wall) ---
try:
    import xgboost as xgb
    print(f"xgboost : {xgb.__version__}")
except ImportError:
    print("xgboost not found — installing...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost>=2.0"],
                   check=True)
    import xgboost as xgb
    print(f"xgboost : {xgb.__version__} (just installed)")

# --- LightGBM (used in backbone stacking) ---
try:
    import lightgbm as lgb
    print(f"lightgbm: {lgb.__version__}")
except ImportError:
    print("lightgbm not found — installing...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm>=4.0"],
                   check=True)
    import lightgbm as lgb
    print(f"lightgbm: {lgb.__version__} (just installed)")

# --- Other deps ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pyarrow>=14.0", "pyyaml>=6.0", "scikit-learn>=1.4",
                "matplotlib>=3.7"], check=True)

print("All dependencies satisfied.")

## Cell 3 — Clone repo (brings src/ AND data/ — no Drive, no gdown)

The dataset `data/CA-PFAS-ASGWS.parquet` is **versioned in the repo** and arrives  
automatically with the clone. This cell is idempotent: if the repo dir already  
exists it only checks out the requested ref.

**Anti-stale-code guard:** after bootstrap, verifies that the cloned code at `GIT_REF`  
contains all required V2 symbols (in the CORRECT files — no cross-file confusion).  
If any are missing, the notebook stops with an explicit message telling you to push  
the missing files and update `GIT_REF`.

In [ ]:
import os, sys, pathlib

REPO_DIR = "/content/pfas-gnn" if IN_COLAB else os.getcwd()

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        print(f"Cloning {REPO_URL} into {REPO_DIR} ...")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print(f"Checking out {GIT_REF} ...")
    subprocess.run(["git", "-C", REPO_DIR, "checkout", GIT_REF], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"workdir : {os.getcwd()}")

# ---- Anti-stale-code guard: v2_fusion_t1 ----
# Checks symbols IN THE CORRECT FILE to avoid cross-file confusion.
V2_T1_PATH = pathlib.Path(REPO_DIR) / "src" / "v2_fusion_t1.py"
assert V2_T1_PATH.exists(), (
    f"FATAL: src/v2_fusion_t1.py not found in the cloned repo at GIT_REF={GIT_REF!r}.\n"
    "You must commit and push the V2 module BEFORE running this notebook on Colab:\n"
    "  git add src/v2_fusion_t1.py src/v2_fusion_gating.py src/hgt_fusion_stacking_t1.py "
    "notebooks/v2_fusion_colab.ipynb\n"
    "  git commit -m 'feat(v2): non-destructive fusion variants for T1a'\n"
    "  git push\n"
    "Then set GIT_REF to the resulting branch or commit SHA and re-run this cell."
)
_v2_src = V2_T1_PATH.read_text()
# Symbols that MUST live in src/v2_fusion_t1.py
for symbol in ["def run(", "def fusion_full64d(", "def fusion_pca_fixed_k(",
               "def fusion_gating_xgb("]:
    assert symbol in _v2_src, (
        f"Symbol {symbol!r} not found in src/v2_fusion_t1.py — "
        f"the repo at GIT_REF={GIT_REF!r} may point to an incomplete commit.\n"
        "Push the complete module and update GIT_REF."
    )
    print(f"  v2_fusion_t1     : {symbol!r:<45} OK")

# ---- Anti-stale-code guard: v2_fusion_gating ----
V2G_PATH = pathlib.Path(REPO_DIR) / "src" / "v2_fusion_gating.py"
assert V2G_PATH.exists(), (
    f"FATAL: src/v2_fusion_gating.py not found at GIT_REF={GIT_REF!r}.\n"
    "Commit and push as shown above."
)
_v2g_src = V2G_PATH.read_text()
# Symbols that MUST live in src/v2_fusion_gating.py (NOT in v2_fusion_t1.py)
for symbol in ["def get_fusion_inputs(", "def train_gating_oof(", "class GatingOOFResult"]:
    assert symbol in _v2g_src, (
        f"Symbol {symbol!r} not found in src/v2_fusion_gating.py — "
        f"the repo at GIT_REF={GIT_REF!r} may point to an incomplete commit."
    )
    print(f"  v2_fusion_gating : {symbol!r:<45} OK")

# ---- Anti-stale-code guard: hgt_fusion_stacking_t1 ----
FB_PATH = pathlib.Path(REPO_DIR) / "src" / "hgt_fusion_stacking_t1.py"
assert FB_PATH.exists(), (
    f"FATAL: src/hgt_fusion_stacking_t1.py not found at GIT_REF={GIT_REF!r}."
)
_fb_src = FB_PATH.read_text()
# Symbols that MUST live in src/hgt_fusion_stacking_t1.py (NOT in other files)
for symbol in ["def oof_embeddings_and_tabular(",
               "def build_oof_backbone(",
               "def stacking_oof_proba("]:
    assert symbol in _fb_src, (
        f"Symbol {symbol!r} not found in src/hgt_fusion_stacking_t1.py — "
        f"the repo at GIT_REF={GIT_REF!r} may be outdated.\n"
        "Push the modified hgt_fusion_stacking_t1.py and update GIT_REF."
    )
    print(f"  hgt_fusion_stacking_t1: {symbol!r:<40} OK")

# ---- Dataset present ----
assert os.path.exists(DATA_PATH), (
    f"FATAL: dataset missing at {DATA_PATH}.\n"
    "The parquet file should be versioned in the repo at data/CA-PFAS-ASGWS.parquet.\n"
    "Check REPO_URL and GIT_REF — the clone may have failed or the file is not committed."
)
print(f"\ndataset : {DATA_PATH}  ({os.path.getsize(DATA_PATH) // 1024} KB)")
print("Code + dataset guard PASSED.")

## Cell 4 — Load dataset & integrity check

Expected shape: **46338 rows x 201 columns**, **11333 unique wells**.  
Required columns: `gm_well_id`, `latitude`, `longitude`, `PFOA_ngL`.  
Any mismatch triggers an explicit stop — no silent failure downstream.

In [ ]:
import pandas as pd

EXPECTED_ROWS  = 46338
EXPECTED_COLS  = 201
EXPECTED_WELLS = 11333
KEY_COLS = ["gm_well_id", "latitude", "longitude", "PFOA_ngL"]

df_probe = pd.read_parquet(DATA_PATH)
n_rows, n_cols = df_probe.shape
n_wells = df_probe["gm_well_id"].nunique()
missing_keys = [c for c in KEY_COLS if c not in df_probe.columns]

print(f"Shape  : {n_rows} rows x {n_cols} cols")
print(f"Wells  : {n_wells} unique")
print(f"Memory : {df_probe.memory_usage(deep=True).sum() // 1024 // 1024} MB")

if missing_keys:
    raise RuntimeError(
        f"FATAL: key columns missing from dataset: {missing_keys}.\n"
        f"Check DATA_PATH={DATA_PATH!r} and GIT_REF={GIT_REF!r}.\n"
        "The parquet must contain: " + ", ".join(KEY_COLS)
    )
print(f"Key columns {KEY_COLS} — all present")

if not SMOKE_TEST:
    if n_rows != EXPECTED_ROWS:
        raise RuntimeError(
            f"FATAL: row count mismatch — got {n_rows}, expected {EXPECTED_ROWS}.\n"
            f"Check GIT_REF={GIT_REF!r} and DATA_PATH={DATA_PATH!r}."
        )
    if n_cols != EXPECTED_COLS:
        raise RuntimeError(
            f"FATAL: column count mismatch — got {n_cols}, expected {EXPECTED_COLS}.\n"
            f"Check GIT_REF={GIT_REF!r}."
        )
    if n_wells != EXPECTED_WELLS:
        raise RuntimeError(
            f"FATAL: well count mismatch — got {n_wells}, expected {EXPECTED_WELLS}.\n"
            f"Check GIT_REF={GIT_REF!r} and DATA_PATH={DATA_PATH!r}."
        )
    print(f"Integrity check PASSED: {n_rows} x {n_cols}, {n_wells} wells.")
else:
    print(f"SMOKE_TEST=True — shape check relaxed (got {n_rows} x {n_cols}, {n_wells} wells).")
    print("  V2.run(smoke=True) will subsample to 500 wells internally.")

del df_probe  # free memory; src.data.load() reloads from DATA_PATH

## Cell 5 — Smoke-test (`SMOKE_TEST=True` only)

Calls `V2.run(smoke=True, compute_delta=False, ...)` which:
- subsamples 500 wells, 3 spatial blocks, 15 epochs (patience 6),
- runs all 5 V2 fusion variants + 3 reference columns (spatial arm only),
- trains the gating MLP nested-LOBO and writes §3.8 curves,
- asserts `cross_block == 0`, all variant AUCs finite, all figures written,
- writes `metrics.json` to `experiments/v2_fusion_smoke/`.

When `SMOKE_TEST=False` this cell is skipped.

In [ ]:
import time, json, math
from pathlib import Path
import numpy as np

if SMOKE_TEST:
    print("=" * 65)
    print("SMOKE-TEST: 500 wells / 3 blocks / 15 epochs — CPU run")
    print("5 fusion variants + 3 references (spatial arm only)")
    print("=" * 65)

    from src import v2_fusion_t1 as V2

    t0 = time.time()
    smoke_out = V2.run(
        smoke=True,
        compute_delta=False,    # skip random arm in smoke to save time
        write=True,
        exp_dir=SMOKE_EXP_DIR,
        seed=SEED,
        verbose=True,
    )
    elapsed_smoke = time.time() - t0

    # ---- end-to-end assertions ----
    sp = smoke_out["spatial"]

    # 1. cross_block == 0 (C-SPAT.2/5)
    n_cross = smoke_out["meta"]["n_cross_block_total"]
    assert n_cross == 0, (
        f"LEAK DETECTED: {n_cross} cross-block edges remain! "
        "Spatial-leakage violation C-SPAT.2/5."
    )
    print(f"  n_cross_block_total = 0  OK")

    # 2. All 5 fusion variants + 3 references present with finite AUC
    EXPECTED_VARIANTS = ["v2a_full64d", "v2b_pca8", "v2b_pca16",
                         "v2c_gating", "v2c_gating_xgb",
                         "xgb_seul", "stacking_v0", "fusion_pca95_v0"]
    for k in EXPECTED_VARIANTS:
        assert k in sp, (
            f"SMOKE FAIL: variant/reference '{k}' missing from output['spatial']. "
            "Check that run() computes all variants + references."
        )
        auc = sp[k]["metrics"]["roc_auc"]
        assert math.isfinite(auc), f"SMOKE FAIL [{k}]: AUC is not finite ({auc})"
        print(f"  [{k:<22}] AUC={auc:.4f}  OK")

    # 3. Gating fold histories non-empty (§3.8)
    hist = smoke_out.get("gating_fold_histories", [])
    assert len(hist) > 0, "SMOKE FAIL: gating fold_histories is empty (§3.8 violation)"
    for h in hist:
        ne = h["n_epochs_ran"]
        assert ne > 0, f"SMOKE FAIL gating fold {h['fold']}: 0 epochs ran"
        assert len(h["history_train_loss"]) == ne, (
            f"gating fold {h['fold']}: train-loss history length {len(h['history_train_loss'])} != {ne}"
        )
        assert len(h["history_val_auc"]) == ne, (
            f"gating fold {h['fold']}: val-auc history length {len(h['history_val_auc'])} != {ne}"
        )
    print(f"  gating fold histories OK "
          f"({len(hist)} folds, epochs={[h['n_epochs_ran'] for h in hist]})")

    # 4. Figures written
    smoke_dir = Path(SMOKE_EXP_DIR)
    for fig_name in ["gating_training_curves.png", "fold_auc_comparison.png"]:
        fig_path = smoke_dir / "figures" / fig_name
        assert fig_path.exists(), (
            f"SMOKE FAIL: figures/{fig_name} not written to {SMOKE_EXP_DIR}/.\n"
            "Check that matplotlib is installed and plot functions ran."
        )
        print(f"  figures/{fig_name}  OK")

    # 5. metrics.json written and parseable
    mpath = smoke_dir / "metrics.json"
    assert mpath.exists(), f"SMOKE FAIL: metrics.json not written to {SMOKE_EXP_DIR}/"
    with open(mpath) as f:
        loaded = json.load(f)
    assert "spatial" in loaded, "metrics.json missing 'spatial' key"
    print("  metrics.json written and parseable  OK")

    print()
    print(f"Smoke test elapsed: {elapsed_smoke:.0f}s ({elapsed_smoke/60:.1f} min)")

    # ---- full-run duration estimate (extrapolation) ----
    # Smoke: 500 wells, 3 blocks, 15 epochs, spatial only
    # Full: ~11333 wells, 8 blocks, 400 epochs, spatial + random (2 backbones)
    scale_epochs = 400 / 15
    scale_blocks = 8 / 3
    scale_regimes = 2   # spatial + random Δ arm
    # GNN scales sublinearly with nodes due to neighbour sampling (~sqrt)
    scale_wells = (11333 / 500) ** 0.5
    est_cpu_s = elapsed_smoke * scale_epochs * scale_blocks * scale_regimes * scale_wells
    est_gpu_s = est_cpu_s / 12   # T4 GPU ~12x speedup (conservative for GNN part)
    print(f"Full-run wall-time estimate (CPU)   : {est_cpu_s/60:.0f} min ({est_cpu_s/3600:.1f} h)")
    print(f"Full-run wall-time estimate (T4 GPU): {est_gpu_s/60:.0f} min ({est_gpu_s/3600:.1f} h)")
    if est_gpu_s > 90 * 60:
        print()
        print("WARNING: estimated GPU time > 90 min.")
        print("Consider COMPUTE_DELTA=False to run only the spatial arm (~half the budget).")
    else:
        print(f"  -> Fits within a Colab T4 session (~25-35 min expected on T4).")

    print()
    print("SMOKE-TEST PASSED. Set SMOKE_TEST=False and switch to a T4 GPU runtime for the full run.")

else:
    print("SMOKE_TEST=False — smoke cell skipped. Proceed to Cell 6 for the full GPU run.")

## Cell 6 — Full run (`SMOKE_TEST=False`)

Calls `V2.run(smoke=False, ...)` which:
1. Builds the **SPATIAL OOF backbone** (8-fold HGT + XGB + LGBM, ~12-18 min on T4).  
   Computes all 5 fusion variant heads + 3 references on the spatial OOF arrays.
2. Trains the **gating MLP** nested-LOBO (§3.8 curves written incrementally).
3. Builds the **RANDOM OOF backbone** (Δ arm, same budget) if `COMPUTE_DELTA=True`.
4. Runs **paired tests** (Nadeau-Bengio + Wilcoxon) vs XGB wall and vs fusion-PCA95-V0.
5. Writes **figures**: `gating_training_curves.png`, `fold_auc_comparison.png`,  
   `spatial_vs_random_scatter.png` to `experiments/v2_fusion/figures/`.
6. Writes `metrics.json`, `REPORT.md`, `config.yaml` to `experiments/v2_fusion/`.

The anti-leak assertion (`n_cross_block_total == 0`) is verified after the run.

In [ ]:
import time
from pathlib import Path

if not SMOKE_TEST:
    print("=" * 65)
    print("FULL RUN: V2 non-destructive fusion variants, T1a")
    print("=" * 65)
    print(f"  hidden={FULL_HIDDEN}  layers={FULL_LAYERS}  heads={FULL_HEADS}  dropout={FULL_DROPOUT}")
    print(f"  max_epochs={FULL_MAX_EPOCHS}  patience={FULL_PATIENCE}  lr={FULL_LR}  wd={FULL_WEIGHT_DECAY}")
    print(f"  n_blocks={FULL_N_BLOCKS}  compute_delta={COMPUTE_DELTA}  seed={SEED}")
    print()
    print("Variants (spatial arm):")
    print("  v2a_full64d    — XGBoost on [tabular || HGT-emb-64D] (no PCA)")
    print("  v2b_pca8       — XGBoost on [tabular || PCA(emb, k=8)]")
    print("  v2b_pca16      — XGBoost on [tabular || PCA(emb, k=16)]")
    print("  v2c_gating     — learned gating MLP (OOF nested-LOBO, §3.8 curves)")
    print("  v2c_gating_xgb — XGBoost on gating fused_repr (OOF)")
    print("References: xgb_seul, stacking_v0, fusion_pca95_v0")
    print()
    if COMPUTE_DELTA:
        print("COMPUTE_DELTA=True: random OOF backbone will be built (adds ~12-18 min).")
    else:
        print("COMPUTE_DELTA=False: only spatial arm (faster, but no Δ diagnostic).")
    print()
    print("gating §3.8 curves written to experiments/v2_fusion/figures/ incrementally.")
    print()

    from src import v2_fusion_t1 as V2

    t0 = time.time()
    full_out = V2.run(
        smoke=False,
        n_blocks=FULL_N_BLOCKS,
        hidden=FULL_HIDDEN,
        layers=FULL_LAYERS,
        dropout=FULL_DROPOUT,
        heads=FULL_HEADS,
        max_epochs=FULL_MAX_EPOCHS,
        patience=FULL_PATIENCE,
        lr=FULL_LR,
        weight_decay=FULL_WEIGHT_DECAY,
        compute_delta=COMPUTE_DELTA,
        write=True,
        exp_dir=EXP_DIR,
        seed=SEED,
        verbose=True,
    )
    elapsed_full = time.time() - t0

    # Verify anti-leak assertion held
    n_cross = full_out["meta"]["n_cross_block_total"]
    assert n_cross == 0, (
        f"LEAK DETECTED in full run: {n_cross} cross-block edges remain. "
        "DO NOT use these results — spatial leakage violates C-SPAT.2/5."
    )
    print(f"\nFull run completed in {elapsed_full:.0f}s ({elapsed_full/60:.1f} min)")
    print(f"Cross-block edges: 0  OK")
    print(f"Artefacts in: {EXP_DIR}/")
    print(f"  metrics.json, REPORT.md, config.yaml")
    print(f"  figures/gating_training_curves.png")
    print(f"  figures/fold_auc_comparison.png")
    if COMPUTE_DELTA:
        print(f"  figures/spatial_vs_random_scatter.png")

else:
    full_out = None
    print("SMOKE_TEST=True — full run cell skipped.")

## Cell 7 — Display results

Reads `experiments/v2_fusion/metrics.json` and renders:
1. **Comparison table** — all 5 V2 variants + 3 references: AUC OOF (spatial), per-fold-mean, CI95, gain vs XGB wall, Δ(random-spatial), PR-AUC, Brier, ECE, NB-p, Wc-p, verdict.
2. **Encadre: la PCA etait-elle le coupable?** — do non-destructive fusions beat fusion_pca95_v0? Robustly?
3. **Paired tests vs V0 (fusion_pca95_v0)** — gain, NB-p, Wc-p, verdict for each V2 variant.
4. **Three figures displayed inline** — gating curves, fold-AUC bar, spatial-vs-random scatter.
5. Full `REPORT.md` text.

Reading from disk is safe even after a reconnect (does not require `full_out` in memory).

In [ ]:
import json, math
from pathlib import Path
import numpy as np

_exp_root = Path(EXP_DIR) if not SMOKE_TEST else Path(SMOKE_EXP_DIR)
metrics_path = _exp_root / "metrics.json"
report_path  = _exp_root / "REPORT.md"

if not metrics_path.exists():
    print(f"metrics.json not found at {metrics_path}.")
    print("Run Cell 5 (smoke) or Cell 6 (full run) first.")
else:
    out  = json.loads(metrics_path.read_text())
    meta = out["meta"]
    sp   = out["spatial"]
    rd   = out.get("random", {})
    cmp  = out.get("comparison", {})
    dlt  = out.get("delta_random_minus_spatial", {})

    print(f"smoke={meta['smoke']}  seed={meta['seed']}  blocks={meta['n_blocks']}  "
          f"wells={meta['n_wells']}  tabular_dim={meta['n_tabular_features']}  "
          f"hgt_emb_dim={meta['hgt_embed_dim']}")
    print(f"elapsed={meta.get('elapsed_s', float('nan')):.0f}s  "
          f"cross_block_edges={meta['n_cross_block_total']}")
    print(f"XGB wall committed : {meta['wall_xgb_spatial_auc_committed']:.4f}  "
          f"RF wall committed : {meta['wall_rf_spatial_auc_committed']:.4f}")
    print()

    def _fmt(v):
        return f"{v:.4f}" if np.isfinite(v) else "n/a"

    # ---- TABLE 1: Main comparison table ----
    print("=" * 100)
    print("TABLE 1 — Comparison: all 5 V2 variants + 3 references (spatial OOF, k={} blocks)".format(
          meta['n_blocks']))
    print("=" * 100)
    hdr = (f"{'variant':<22} {'AUC OOF':>8}  {'pfm AUC':>8}  {'CI95':>17}  "
           f"{'gain vs XGB':>11}  {'D(rnd-sp)':>9}  "
           f"{'PR-AUC':>7}  {'Brier':>7}  {'ECE':>6}  "
           f"{'NB-p':>7}  {'Wc-p':>7}  verdict-vs-XGB")
    print(hdr)
    print("-" * len(hdr))

    ALL_ROWS = ["v2a_full64d", "v2b_pca8", "v2b_pca16",
                "v2c_gating", "v2c_gating_xgb",
                "xgb_seul", "stacking_v0", "fusion_pca95_v0"]

    for name in ALL_ROWS:
        if name not in sp:
            continue
        ev  = sp[name]
        m   = ev["metrics"]
        ci  = ev.get("auc_ci95", {})
        pfm = ev.get("per_fold_mean_auc", float("nan"))
        rd_pfm = rd.get(name, {}).get("per_fold_mean_auc", float("nan"))
        delta_pfm = rd_pfm - pfm if np.isfinite(rd_pfm) else float("nan")

        c = cmp.get(name, {})
        gain = c.get("gain_vs_xgb_wall_per_fold_mean", float("nan"))
        nb_p = c.get("tests_vs_xgb_wall", {}).get("nadeau_bengio", {}).get("p", float("nan"))
        wc_p = c.get("tests_vs_xgb_wall", {}).get("wilcoxon", {}).get("p", float("nan"))
        verdict = c.get("verdict_vs_xgb", "n/a")

        ci_str = f"[{_fmt(ci.get('ci_low', float('nan')))},{_fmt(ci.get('ci_high', float('nan')))}]"
        print(f"{name:<22} {_fmt(m['roc_auc']):>8}  {_fmt(pfm):>8}  {ci_str:>17}  "
              f"{gain:>+11.4f}  {delta_pfm:>+9.4f}  "
              f"{_fmt(m.get('pr_auc', float('nan'))):>7}  "
              f"{_fmt(m.get('brier', float('nan'))):>7}  "
              f"{_fmt(m.get('ece', float('nan'))):>6}  "
              f"{_fmt(nb_p):>7}  {_fmt(wc_p):>7}  {verdict}")

    print()
    xgb_oof  = sp.get("xgb_seul", {}).get("metrics", {}).get("roc_auc", float("nan"))
    xgb_pfm  = sp.get("xgb_seul", {}).get("per_fold_mean_auc", float("nan"))
    v0_oof   = sp.get("fusion_pca95_v0", {}).get("metrics", {}).get("roc_auc", float("nan"))
    v0_pfm   = sp.get("fusion_pca95_v0", {}).get("per_fold_mean_auc", float("nan"))
    print(f"In-run XGB wall:     AUC OOF={_fmt(xgb_oof)}  per-fold-mean={_fmt(xgb_pfm)}")
    print(f"V0 fusion-PCA95 ref: AUC OOF={_fmt(v0_oof)}  per-fold-mean={_fmt(v0_pfm)}")
    print(f"Reality rule: robust gain requires p<0.05 AND gain>0.03 AUC per-fold-mean.")

    # ---- ENCADRE: la PCA etait-elle le coupable? ----
    print()
    print("=" * 80)
    print("ENCADRE — La PCA etait-elle le coupable?")
    print("(V0 used PCA-to-95%-var on 64-D HGT emb, kept ~47-48 components, yet fell")
    print(" below XGB wall. V2 non-destructive fusions answer: was PCA the bottleneck?)")
    print("=" * 80)
    print()
    v2_variants = ["v2a_full64d", "v2b_pca8", "v2b_pca16", "v2c_gating", "v2c_gating_xgb"]
    any_beats_v0 = False
    any_robust_beats_v0 = False
    print(f"{'variant':<22} {'AUC pfm vs v0':>14}  {'NB-p vs V0':>11}  {'Wc-p vs V0':>11}  verdict-vs-V0")
    print("-" * 80)
    for name in v2_variants:
        c = cmp.get(name, {})
        gain_v0 = c.get("gain_vs_v0_pca95_per_fold_mean", float("nan"))
        nb_v0   = c.get("tests_vs_fusion_pca95_v0", {}).get("nadeau_bengio", {}).get("p", float("nan"))
        wc_v0   = c.get("tests_vs_fusion_pca95_v0", {}).get("wilcoxon", {}).get("p", float("nan"))
        verd_v0 = c.get("verdict_vs_v0", "n/a")
        if np.isfinite(gain_v0) and gain_v0 > 0:
            any_beats_v0 = True
        if verd_v0 == "robust_gain":
            any_robust_beats_v0 = True
        print(f"{name:<22} {gain_v0:>+14.4f}  {_fmt(nb_v0):>11}  {_fmt(wc_v0):>11}  {verd_v0}")
    print()
    if any_robust_beats_v0:
        print("VERDICT: At least one V2 non-destructive fusion ROBUSTLY beats fusion_pca95_v0")
        print("(p<0.05 AND >0.03 AUC gain). PCA DID contribute to the V0 gap.")
        print("However, check whether it also beats the XGB wall (Table 1) — that is the")
        print("harder bar that determines whether the GNN embedding is truly additive.")
    elif any_beats_v0:
        print("VERDICT: Some V2 variants show positive gains vs V0, but NONE reach the")
        print("robust threshold (p<0.05 AND >0.03 AUC). The PCA contributed marginally")
        print("but is not the primary bottleneck. The HGT embedding may simply not be")
        print("additive for XGBoost on this spatial prediction task.")
    else:
        print("VERDICT: No V2 variant beats fusion_pca95_v0 (V0 reference). The PCA was")
        print("NOT the bottleneck. The HGT embedding itself is not helpful for XGBoost here,")
        print("regardless of how many PCA components are retained.")
        print("This is a reportable, informative result (expected by the project design).")

    # ---- TABLE 2: Paired tests vs V0 ----
    print()
    print("=" * 80)
    print("TABLE 2 — Paired tests vs fusion_pca95_v0 (per-fold-mean, 8 spatial folds)")
    print("=" * 80)
    hdr2 = f"{'variant':<22} {'gain vs V0 pfm':>15}  {'NB p':>8}  {'Wc p':>8}  verdict-vs-V0"
    print(hdr2)
    print("-" * len(hdr2))
    for name in v2_variants:
        c = cmp.get(name, {})
        gain_v0 = c.get("gain_vs_v0_pca95_per_fold_mean", float("nan"))
        nb_v0   = c.get("tests_vs_fusion_pca95_v0", {}).get("nadeau_bengio", {}).get("p", float("nan"))
        wc_v0   = c.get("tests_vs_fusion_pca95_v0", {}).get("wilcoxon", {}).get("p", float("nan"))
        verd_v0 = c.get("verdict_vs_v0", "n/a")
        print(f"{name:<22} {gain_v0:>+15.4f}  {_fmt(nb_v0):>8}  {_fmt(wc_v0):>8}  {verd_v0}")

    # ---- TABLE 3: Delta (random - spatial) ----
    if dlt:
        print()
        print("=" * 80)
        print("TABLE 3 — Delta(random - spatial) per variant (spatial-leakage inflation)")
        print("(above 0 = random CV inflates AUC vs spatial block CV)")
        print("=" * 80)
        hdr3 = f"{'variant':<22} {'spatial AUC pfm':>16}  {'random AUC pfm':>15}  {'delta pfm':>10}"
        print(hdr3)
        print("-" * len(hdr3))
        for name in ALL_ROWS:
            if name not in sp or name not in rd:
                continue
            sp_pfm = sp[name].get("per_fold_mean_auc", float("nan"))
            rd_pfm = rd.get(name, {}).get("per_fold_mean_auc", float("nan"))
            d = dlt.get(name, float("nan"))
            print(f"{name:<22} {_fmt(sp_pfm):>16}  {_fmt(rd_pfm):>15}  {d:>+10.4f}")

    # ---- Gating §3.8 diagnostic ----
    gating_diag = meta.get("gating_diag", {})
    if gating_diag:
        print()
        print("=" * 80)
        print("Gating MLP §3.8 diagnostic (variant c — iterative, per-epoch curves)")
        print("=" * 80)
        print(f"  mean gate value (graph weight vs tabular): "
              f"{gating_diag.get('mean_gate_value', float('nan')):.3f}")
        print(f"  (0.0 = fully tabular, 1.0 = fully graph-embedding)")
        print(f"  mean epochs / fold : {gating_diag.get('mean_epochs', float('nan')):.1f}")
        print(f"  n_epochs per fold  : {gating_diag.get('n_epochs_per_fold', [])}")
        ut = gating_diag.get("undertrained_folds", [])
        if ut:
            print(f"  *** UNDERTRAINED FOLDS: {ut} — check gating_training_curves.png ***")
            print("  ACTION: raise max_epochs and/or patience in the gating head params.")
        else:
            print(f"  undertrained folds : none — early-stop appears stable.")

    # ---- Figures inline ----
    print()
    print("=" * 80)
    print("Figures (CLAUDE.md §3.8)")
    print("=" * 80)
    try:
        import matplotlib.pyplot as plt
        import matplotlib.image as mpimg
        FIGS = [
            ("figures/gating_training_curves.png",
             "Variant (c) gating MLP: val-AUC and train-loss vs epoch per fold (§3.8)"),
            ("figures/fold_auc_comparison.png",
             "All variants: per-fold AUC bar chart (§3.8 diagnostic for XGBoost aval heads)"),
            ("figures/spatial_vs_random_scatter.png",
             "Spatial vs random AUC per fold (spatial-leakage inflation)"),
        ]
        for rel_path, title in FIGS:
            png_path = _exp_root / rel_path
            if png_path.exists():
                img = mpimg.imread(str(png_path))
                fig, ax = plt.subplots(1, 1, figsize=(13, 4.5))
                ax.imshow(img)
                ax.axis("off")
                ax.set_title(title, fontsize=10)
                plt.tight_layout()
                plt.show()
                print(f"  Displayed: {png_path}")
            else:
                print(f"  {rel_path} not found (may not have been produced; e.g. COMPUTE_DELTA=False)")
    except Exception as e:
        print(f"  Could not display PNGs inline: {e}")
        print(f"  Find them at: {_exp_root}/figures/")

    # ---- REPORT.md ----
    if report_path.exists():
        print()
        print("=" * 80)
        print("REPORT.md")
        print("=" * 80)
        print(report_path.read_text())

## Cell 8 — Persist outputs

> **WARNING: the Colab workspace is EPHEMERAL.** All files in `/content/pfas-gnn/`  
> (including `metrics.json`, `REPORT.md`, `figures/*.png`) are **permanently lost**  
> when the runtime disconnects or times out. Run at least one option below.

**Option A (quick):** `files.download()` creates a local zip archive including all figures.  
**Option B (recommended):** `git commit + push` versions the artefacts in the repo.

The zip archive will contain:  
`v2_fusion/metrics.json`, `v2_fusion/REPORT.md`, `v2_fusion/config.yaml`,  
`v2_fusion/figures/gating_training_curves.png`, `v2_fusion/figures/fold_auc_comparison.png`,  
`v2_fusion/figures/spatial_vs_random_scatter.png`.

In [ ]:
import shutil, os
from pathlib import Path

_persist_dir = Path(EXP_DIR) if not SMOKE_TEST else Path(SMOKE_EXP_DIR)

if not _persist_dir.exists():
    print(f"No outputs found at {_persist_dir}. Run Cell 5 or Cell 6 first.")
else:
    files_found = [p for p in sorted(_persist_dir.rglob("*")) if p.is_file()]
    print(f"Outputs in {_persist_dir}/ ({len(files_found)} files):")
    for p in files_found:
        print(f"  {p.relative_to(_persist_dir)}  ({p.stat().st_size // 1024} KB)")

    # ==== Option A: download zip archive (MUST include figures/*.png) ====
    archive_stem = "v2_fusion_outputs"
    archive_path = shutil.make_archive(
        archive_stem, "zip",
        root_dir=str(_persist_dir.parent),
        base_dir=_persist_dir.name
    )
    archive_size = os.path.getsize(archive_path) // 1024
    print(f"\nOption A — Archive: {archive_path} ({archive_size} KB)")
    print("  Contains: metrics.json, REPORT.md, config.yaml,")
    print("            figures/gating_training_curves.png,")
    print("            figures/fold_auc_comparison.png,")
    print("            figures/spatial_vs_random_scatter.png")
    if IN_COLAB:
        from google.colab import files
        files.download(archive_path)
        print("  Download triggered — save this zip to your local machine.")
    else:
        print(f"  (Not in Colab — archive at {os.path.abspath(archive_path)})")

    # ==== Option B: git commit + push ====
    # Uncomment and fill in your git identity if needed.
    # Push artefacts back to the repo for versioning and inter-agent access.
    #
    # import subprocess
    # _smoke_flag = "smoke" if SMOKE_TEST else "full"
    # _exp_name = "v2_fusion_smoke" if SMOKE_TEST else "v2_fusion"
    # # (Optional) set identity if not configured:
    # # subprocess.run(["git", "config", "user.email", "dnjomouloic@gmail.com"], cwd=REPO_DIR)
    # # subprocess.run(["git", "config", "user.name", "Loic"], cwd=REPO_DIR)
    # _to_add = [
    #     f"experiments/{_exp_name}/metrics.json",
    #     f"experiments/{_exp_name}/REPORT.md",
    #     f"experiments/{_exp_name}/config.yaml",
    #     f"experiments/{_exp_name}/figures/gating_training_curves.png",
    #     f"experiments/{_exp_name}/figures/fold_auc_comparison.png",
    #     f"experiments/{_exp_name}/figures/spatial_vs_random_scatter.png",
    # ]
    # subprocess.run(["git", "add"] + _to_add, cwd=REPO_DIR, check=True)
    # subprocess.run(["git", "commit", "-m",
    #     f"results(v2_fusion): {_smoke_flag} run — non-destructive fusion T1a from Colab GPU"
    # ], cwd=REPO_DIR, check=True)
    # subprocess.run(["git", "push"], cwd=REPO_DIR, check=True)
    # print("Option B — Results committed and pushed to", REPO_URL)

    print()
    print("REMINDER: without Option A download OR Option B git push, ALL outputs")
    print("(metrics.json, REPORT.md, figures/*.png) are permanently lost")
    print("when this Colab runtime disconnects or times out.")